# OpenSearch 인덱스 구축

말뭉치 parquet → OpenSearch 인덱싱 파이프라인을 하나로 통합한 노트북.

### 처리 단계
1. (선택: 기존 인덱스 삭제 → )새 인덱스 생성 (전체 스키마 포함)
2. 말뭉치 로드 → 오류 주석 추출 (시그니처, 교정 쌍, 오류 양상)
3. 문장별 임베딩 생성 + 오류 정보와 함께 bulk 인덱싱
4. 검증

In [1]:
import pandas as pd
from opensearchpy import OpenSearch, helpers
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

client = OpenSearch(
    hosts=[{"host": "172.30.1.81", "port": 9200}],
    http_auth=None,
    use_ssl=False,
    verify_certs=False,
)

model = SentenceTransformer("../model/KURE-v1", local_files_only=True)
print(f"임베딩 모델 로드 완료 (dim={model.get_sentence_embedding_dimension()})")
print(f"OpenSearch: {client.info()['version']['number']}")

/home/puju/py_playground/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█| 391/391 [00:01<00:00, 303.78it/s, Materializing param=p


임베딩 모델 로드 완료 (dim=1024)
OpenSearch: 2.18.0


In [2]:
INDEX_NAME = "korean_test_0222"

# 기존 인덱스 삭제
if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)
    print(f"기존 인덱스 '{INDEX_NAME}' 삭제 완료")

# 새 인덱스 생성 (오류 필드 포함 전체 스키마)
index_body = {
    "settings": {
        "index.knn": True,
        "analysis": {
            "tokenizer": {
                "nori_tokenizer": {
                    "type": "nori_tokenizer",
                    "decompound_mode": "mixed",
                }
            },
            "analyzer": {
                "korean": {
                    "type": "custom",
                    "tokenizer": "nori_tokenizer",
                }
            },
        },
    },
    "mappings": {
        "properties": {
            "original_text": {"type": "text", "analyzer": "korean"},
            "morphs": {"type": "keyword"},
            "embedding": {
                "type": "knn_vector",
                "dimension": 1024,
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "engine": "lucene",
                },
            },
            "error_signatures": {"type": "keyword"},
            "correction_pairs": {"type": "keyword"},
            "error_patterns": {"type": "keyword"},
            "has_error": {"type": "boolean"},
        }
    },
}

client.indices.create(index=INDEX_NAME, body=index_body)
print(f"인덱스 '{INDEX_NAME}' 생성 완료")
print(f"매핑: {list(client.indices.get_mapping(index=INDEX_NAME)[INDEX_NAME]['mappings']['properties'].keys())}")

기존 인덱스 'korean_test_0222' 삭제 완료
인덱스 'korean_test_0222' 생성 완료
매핑: ['correction_pairs', 'embedding', 'error_patterns', 'error_signatures', 'has_error', 'morphs', 'original_text']


In [3]:
# 말뭉치 로드
CORPUS_FILE = "말뭉치_2024.parquet.gzip"
df = pd.read_parquet(CORPUS_FILE)
print(f"말뭉치 로드: {len(df):,}행, 고유 문장: {df.groupby(['표본 번호', '문장']).ngroups:,}")

말뭉치 로드: 2,608,912행, 고유 문장: 235,902


In [4]:
# 오류 주석 추출

def make_error_signature(row):
    loc = row["오류 위치"]
    pat = row["오류 양상"]
    lvl = row["오류 층위"] if row["오류 층위"] != "0" else ""
    if lvl:
        return f"{loc}:{pat}:{lvl}"
    return f"{loc}:{pat}"

def make_correction_pair(row):
    orig = row["원 형태소"] if row["원 형태소"] != "0" else "∅"
    orig_tag = row["형태 주석"] if row["형태 주석"] != "0" else ""
    corr = row["교정 형태소"] if row["교정 형태소"] != "0" else "∅"
    corr_tag = row["교정 주석"] if row["교정 주석"] != "0" else ""
    orig_str = f"{orig}/{orig_tag}" if orig_tag else orig
    corr_str = f"{corr}/{corr_tag}" if corr_tag else corr
    return f"{orig_str}→{corr_str}"

err_rows = df[df["오류 양상"] != "0"].copy()
err_rows["signature"] = err_rows.apply(make_error_signature, axis=1)
err_rows["correction_pair"] = err_rows.apply(make_correction_pair, axis=1)

sentence_errors = err_rows.groupby(["표본 번호", "문장"]).agg(
    error_signatures=("signature", list),
    correction_pairs=("correction_pair", list),
    error_patterns=("오류 양상", list),
).reset_index()

# 문장 텍스트 → 오류 정보 딕셔너리
error_by_text = {}
for _, row in sentence_errors.iterrows():
    error_by_text[row["문장"]] = {
        "error_signatures": row["error_signatures"],
        "correction_pairs": row["correction_pairs"],
        "error_patterns": row["error_patterns"],
    }

print(f"오류 행: {len(err_rows):,}")
print(f"오류 문장: {len(sentence_errors):,}")
print(f"\n샘플:")
pd.set_option("display.max_colwidth", 120)
print(sentence_errors[["문장", "error_signatures", "correction_pairs"]].head(5).to_string())

오류 행: 137,298
오류 문장: 65,791

샘플:
                           문장                     error_signatures                       correction_pairs
0      그래서 열두 시에 다 끝난면 자도 돼요.  [CMAJ:REP:DC, CNNG:OM, FED:MIF:MCJ]  [그래서/MAJ→그리고/MAJ, ∅→전/NNG, 면/EC→면/EC]
1      수업이 끝난 후에 친구하고 약속 있어요.                 [FAP:REP:DS, FNP:OM]                [하고/JKB→와/JKB, ∅→이/JKS]
2  혼자 집에서 밥 먹고 텔레비전 보고 재미있어요.           [FOP:REP, FOP:OM, FED:REP]         [∅→을/JKO, ∅→을/JKO, 고/EC→아서/EC]
3   그래서 토요일 저녁에 친구하고 술을 마셨어요.                         [FAP:REP:DS]                         [하고/JKB→와/JKB]
4            그리고 이야가도 많이 했어요.                           [CNNG:MIF]                      [이야가/NNG→이야기/NNG]


In [ ]:
# 문장별 그루핑 → 임베딩 + 오류 정보와 함께 bulk 인덱싱
grouped = df.groupby(["표본 번호", "문장"])

BATCH_SIZE = 200
batch = []
total_indexed = 0

for idx, ((sample_id, sentence), group) in enumerate(tqdm(grouped, total=len(grouped), desc="인덱싱")):
    # 형태소 배열
    morphs = [row["형태 주석"] for _, row in group.iterrows() if row["형태 주석"] != "0"]

    # 임베딩
    embedding = model.encode(sentence).tolist()

    # 오류 정보
    err = error_by_text.get(sentence)
    if err:
        error_signatures = err["error_signatures"]
        correction_pairs = err["correction_pairs"]
        error_patterns = err["error_patterns"]
        has_error = True
    else:
        error_signatures = []
        correction_pairs = []
        error_patterns = []
        has_error = False

    batch.append({
        "_index": INDEX_NAME,
        "_id": f"{sample_id}_{idx}",
        "_source": {
            "original_text": sentence,
            "morphs": morphs,
            "embedding": embedding,
            "error_signatures": error_signatures,
            "correction_pairs": correction_pairs,
            "error_patterns": error_patterns,
            "has_error": has_error,
        },
    })

    if len(batch) >= BATCH_SIZE:
        success, _ = helpers.bulk(client, batch, raise_on_error=False)
        total_indexed += success
        batch = []

# 잔여 배치
if batch:
    success, _ = helpers.bulk(client, batch, raise_on_error=False)
    total_indexed += success

client.indices.refresh(index=INDEX_NAME)
print(f"\n인덱싱 완료: {total_indexed:,}건")

인덱싱:   0%|▏                                                                                                            | 358/235902 [00:13<1:53:36, 34.55it/s]

In [3]:
# 검증
total = client.count(index=INDEX_NAME)["count"]
errors = client.count(index=INDEX_NAME, body={"query": {"term": {"has_error": True}}})["count"]
normal = client.count(index=INDEX_NAME, body={"query": {"term": {"has_error": False}}})["count"]

print(f"전체 문서: {total:,}")
print(f"오류 문서: {errors:,}")
print(f"정상 문서: {normal:,}")

# 샘플 확인
resp = client.search(
    index=INDEX_NAME,
    body={"size": 3, "query": {"term": {"has_error": True}},
          "_source": ["original_text", "error_signatures", "correction_pairs", "has_error"]},
)
print("\n--- 오류 문서 샘플 ---")
for hit in resp["hits"]["hits"]:
    src = hit["_source"]
    print(f"  [{hit['_id']}] {src['original_text']}")
    print(f"    시그니처: {src['error_signatures']}")
    print(f"    교정 쌍:  {src['correction_pairs']}")

전체 문서: 235,902
오류 문서: 70,078
정상 문서: 165,824

--- 오류 문서 샘플 ---
  [56_0] 그래서 열두 시에 다 끝난면 자도 돼요.
    시그니처: ['CMAJ:REP:DC', 'CNNG:OM', 'FED:MIF:MCJ']
    교정 쌍:  ['그래서/MAJ→그리고/MAJ', '∅→전/NNG', '면/EC→면/EC']
  [56_6] 수업이 끝난 후에 친구하고 약속 있어요.
    시그니처: ['FAP:REP:DS', 'FNP:OM']
    교정 쌍:  ['하고/JKB→와/JKB', '∅→이/JKS']
  [56_10] 혼자 집에서 밥 먹고 텔레비전 보고 재미있어요.
    시그니처: ['FOP:REP', 'FOP:OM', 'FED:REP']
    교정 쌍:  ['∅→을/JKO', '∅→을/JKO', '고/EC→아서/EC']
